In [0]:
# 1. Fetch the last recorded watermark value for here_tech
watermark_df = spark.sql("""
    SELECT last_watermark_value 
    FROM inlap.control.watermark_table 
    WHERE source_vendor = 'here_tech'
""")

# Extract it into a Python variable
last_watermark = watermark_df.collect()[0]["last_watermark_value"]
print(f"Filtering records newer than watermark: {last_watermark}")

Filtering records newer than watermark: 2025-07-17 00:00:00


In [0]:
%sql
create schema if not exists inlap.bronze

In [0]:
%sql
DROP TABLE IF EXISTS inlap.bronze.heretech;

In [0]:
from pyspark.sql.functions import col, current_timestamp, from_utc_timestamp, to_timestamp, regexp_replace

# 1. Read raw CSV
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(
        "abfss://datalake@attinlapsa.dfs.core.windows.net/vendors/here_tech.csv"
    )
)

In [0]:
# 2. Apply all transformations including timestamps and metadata
df = (
    df.withColumn(
        "_ingestion_timestamp",
        from_utc_timestamp(current_timestamp(), "Asia/Kolkata"),
    )
    .withColumn("source_file", col("_metadata.file_path"))\
    .withColumn("source_name", regexp_replace(col("_metadata.file_name"), "\\.csv$", ""))
)

In [0]:
df=df.drop("updated_at")

In [0]:
from delta.tables import DeltaTable

table_path = "abfss://datalake@attinlapsa.dfs.core.windows.net/bronze/heretech/"
table_name = "inlap.bronze.heretech"

# Check if the Delta table already exists
if DeltaTable.isDeltaTable(spark, table_path):
    print("Table exists. Appending incremental data...")
    # For subsequent runs, write directly to the registered Unity Catalog table name
    (
        df.write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )
else:
    print("Table does not exist. Initializing table for the first time...")
    # For the very first run, specify the path to initialize it
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .option("path", table_path)
        .saveAsTable(table_name)
    )

Table exists. Appending incremental data...


In [0]:
from pyspark.sql.functions import max as spark_max

# 1. Calculate the max timestamp from your incoming dataframe batch
new_watermark = df.agg(spark_max("last_updated")).collect()[0][0]
new_watermark_str = str(new_watermark)

# 2. Execute the watermark update in your control table
spark.sql(f"""
    UPDATE inlap.control.watermark_table 
    SET last_watermark_value = '{new_watermark_str}',
        last_run_timestamp = current_timestamp(),
        status = 'SUCCESS'
    WHERE source_vendor = 'here_tech'
""")

print(f"Watermark table successfully updated to: {new_watermark_str}")

Watermark table successfully updated to: 2025-07-17


In [0]:
import datetime

# 1. Capture execution metrics
run_timestamp = datetime.datetime.now()
vendor_name = "here_tech"
layer_name = "bronze"

# 2. Log to a central audit tracking table (create this table once if you haven't already)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS inlap.control.audit_log (
        source_vendor STRING,
        layer STRING,
        run_timestamp TIMESTAMP,
        status STRING
    ) USING DELTA;
""")

# 3. Insert the success audit record
spark.sql(f"""
    INSERT INTO inlap.control.audit_log 
    VALUES ('{vendor_name}', '{layer_name}', current_timestamp(), 'SUCCESS')
""")

print("Step 6: Audit log successfully recorded for this run.")

Step 6: Audit log successfully recorded for this run.
